# SANAD — Stage 3 FINAL FULL VERSION
## LLM Generation, Grounding Evaluation & Citation Verification

### Final upstream pipeline used by this notebook

**Stage 1**  
Legal documents → cleaning / legal-article segmentation → BGE-M3 embeddings → FAISS

**Stage 2 — Experiment B**  
BGE-M3 + existing FAISS index → Dense vs BM25 vs Hybrid → **selected retrieval** → Top-10

**Stage 2 — Experiment C**  
Top-10 from Experiment B → BGE reranker vs Qwen reranker → **selected reranker** → Top-3

**Stage 3 — this notebook**  
Top-3 from Experiment C → augmented legal context → ALLaM vs Qwen3 generation → citation verification → answer-quality evaluation → final generator selection

> **Important:** Stage 3 does **not** rebuild embeddings, run FAISS retrieval, run BM25, or rerank documents.




## 1. Stage 3 Goal

Stage 3 compares two generators using the **same 60 questions, the same Top-3 legal articles, and the same prompt**.

Compared generators:
- `humain-ai/ALLaM-7B-Instruct-preview`
- `Qwen/Qwen3-8B`

Evaluation includes:
- Correctness
- Faithfulness
- Citation accuracy
- Citation precision against retrieved context
- Citation hallucination
- Arabic quality
- Generation time
- Valid JSON rate

The retrieved Top-3 are treated as fixed input from Experiment C.

## 2. Required Input Files

Upload only:

1. `SANAD_Experiment_C_Top3_for_Stage3.csv`  
   Direct output from Experiment C. Expected: **60 queries × 3 rows = 180 rows**.

2. `saudi_legal_moj_clean.parquet`  
   Used only to recover the gold legal article text for evaluation. It is **not used for retrieval** in Stage 3.

## 3. Install libraries

In [ ]:
!pip install -q -U "transformers>=4.51.0" accelerate bitsandbytes sentencepiece "protobuf>=7.36.1" pyarrow pandas tqdm

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 7.36.2 which is incompatible.


In [ ]:
!pip install -U "protobuf>=7.36.1" sentencepiece

## 4. Imports, reproducibility, GPU, and checkpoint folder

In [ ]:
import os
import re
import gc
import json
import time
import shutil
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    set_seed,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
set_seed(SEED)

assert torch.cuda.is_available(), (
    "GPU is required. In Colab: Runtime → Change runtime type → T4 GPU."
)

print("GPU:", torch.cuda.get_device_name(0))

# Keep checkpoints so Colab disconnections do not lose completed generations.
USE_DRIVE_CHECKPOINT = True

if USE_DRIVE_CHECKPOINT:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_DIR = Path("/content/drive/MyDrive/SANAD_Stage3_REBUILT")
else:
    WORK_DIR = Path("/content/SANAD_Stage3_REBUILT")

WORK_DIR.mkdir(parents=True, exist_ok=True)
print("Working directory:", WORK_DIR)

GPU: Tesla T4
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working directory: /content/drive/MyDrive/SANAD_Stage3_REBUILT


## 5. Upload or restore the two required input files

In [ ]:
from google.colab import files

required_files = [
    "SANAD_Experiment_C_Top3_for_Stage3.csv",
    "saudi_legal_moj_clean.parquet",
]

for filename in required_files:
    content_path = Path("/content") / filename
    saved_path = WORK_DIR / filename

    if (not content_path.exists()) and saved_path.exists():
        shutil.copy(saved_path, content_path)
        print("Restored:", filename)

missing = [f for f in required_files if not (Path("/content") / f).exists()]

if missing:
    print("Please upload:", missing)
    files.upload()

for filename in required_files:
    src = Path("/content") / filename
    assert src.exists(), f"Missing required file: {filename}"

    dst = WORK_DIR / filename
    if not dst.exists():
        shutil.copy(src, dst)

print("✅ Required files are ready.")

✅ Required files are ready.


## 6. Load and validate Experiment C Top-3

This validation guarantees that Stage 3 receives a clean handoff:
- 180 rows
- 60 unique questions
- exactly 3 candidates per question
- rerank ranks 1, 2, 3
- one consistent selected retrieval method
- one consistent selected reranker

No retrieval or reranking is repeated here.

In [ ]:
TOP3_PATH = "/content/SANAD_Experiment_C_Top3_for_Stage3.csv"
KB_PATH = "/content/saudi_legal_moj_clean.parquet"

top3_df = pd.read_csv(
    TOP3_PATH,
    dtype={
        "query_id": str,
        "query_type": str,
        "query": str,
        "gold_law": str,
        "gold_article": str,
        "selected_retrieval": str,
        "selected_reranker": str,
        "law_name": str,
        "article_number": str,
        "doc_text": str,
    }
)

kb_df = pd.read_parquet(KB_PATH).reset_index(drop=True)

required_columns = {
    "query_id",
    "query_type",
    "query",
    "gold_law",
    "gold_article",
    "selected_retrieval",
    "selected_reranker",
    "rerank_rank",
    "law_name",
    "article_number",
    "doc_text",
}

missing_cols = required_columns - set(top3_df.columns)
assert not missing_cols, f"Missing columns in Experiment C output: {missing_cols}"

assert len(top3_df) == 180, f"Expected 180 Top-3 rows, found {len(top3_df)}."
assert top3_df["query_id"].nunique() == 60, (
    f"Expected 60 unique queries, found {top3_df['query_id'].nunique()}."
)

counts = top3_df.groupby("query_id").size()
assert counts.eq(3).all(), "Each query must have exactly 3 rows."

rank_sets = top3_df.groupby("query_id")["rerank_rank"].apply(
    lambda s: set(pd.to_numeric(s, errors="coerce").dropna().astype(int))
)
assert rank_sets.apply(lambda x: x == {1, 2, 3}).all(), (
    "Each query must contain rerank ranks 1, 2, and 3."
)

retrieval_values = sorted(
    top3_df["selected_retrieval"].dropna().astype(str).str.strip().unique()
)
reranker_values = sorted(
    top3_df["selected_reranker"].dropna().astype(str).str.strip().unique()
)

assert len(retrieval_values) == 1, (
    f"Expected one selected retrieval method, found: {retrieval_values}"
)
assert len(reranker_values) == 1, (
    f"Expected one selected reranker, found: {reranker_values}"
)

SELECTED_RETRIEVAL = retrieval_values[0]
SELECTED_RERANKER = reranker_values[0]

print("Rows:", len(top3_df))
print("Queries:", top3_df["query_id"].nunique())
print("Selected retrieval:", SELECTED_RETRIEVAL)
print("Selected reranker:", SELECTED_RERANKER)
print("Knowledge-base rows:", len(kb_df))
print("✅ Experiment C Top-3 handoff verified.")

Rows: 180
Queries: 60
Selected retrieval: Dense
Selected reranker: BAAI/bge-reranker-v2-m3
Knowledge-base rows: 4226
✅ Experiment C Top-3 handoff verified.


## 7. Build one augmented legal context per query

Experiment C stores one row per retrieved article.  
Stage 3 converts the 3 rows into one context block for each question.

The output of this cell is **60 rows**, one per query.

In [ ]:
def build_context_for_group(group):
    group = group.sort_values("rerank_rank")

    parts = []

    for _, row in group.iterrows():
        rank = int(row["rerank_rank"])

        parts.append(
            f"[Retrieved Article {rank}]\n"
            f"Law: {str(row['law_name']).strip()}\n"
            f"Article: {str(row['article_number']).strip()}\n"
            f"Text:\n{str(row['doc_text']).strip()}"
        )

    return "\n\n".join(parts)


context_rows = []

for query_id, group in top3_df.groupby("query_id", sort=False):
    first = group.iloc[0]

    context_rows.append({
        "query_id": str(query_id),
        "query_type": str(first["query_type"]),
        "query": str(first["query"]),
        "gold_law": str(first["gold_law"]),
        "gold_article": str(first["gold_article"]),
        "selected_retrieval": str(first["selected_retrieval"]),
        "selected_reranker": str(first["selected_reranker"]),
        "augmented_context": build_context_for_group(group),
    })

context_df = pd.DataFrame(context_rows)

assert len(context_df) == 60
assert context_df["query_id"].nunique() == 60

CONTEXT_60_PATH = WORK_DIR / "SANAD_Stage3_Context_60.csv"
context_df.to_csv(CONTEXT_60_PATH, index=False, encoding="utf-8-sig")

print("Stage 3 contexts:", len(context_df))
print("Saved:", CONTEXT_60_PATH)
print("\nExample context:\n")
print(context_df.loc[0, "augmented_context"][:2500])

Stage 3 contexts: 60
Saved: /content/drive/MyDrive/SANAD_Stage3_REBUILT/SANAD_Stage3_Context_60.csv

Example context:

[Retrieved Article 1]
Law: نظام التكاليف القضائية
Article: المادة الثانية
Text:
اسم النظام: نظام التكاليف القضائية
رقم المادة: المادة الثانية
النص القانوني: المادة الثانية
تسري أحكام النظام على جميع الدعاوى والطلبات التي تقدم إلى المحاكم» فيما عدا مايأتي:
١.الدعاوى الجزائية العامة»والدعاوى التأديبية»والطلبات المتعلقبها.
؟.الدعاوى والطلبات التي تختص بها محاكم الأحوال الشخصية» عدا طلب النقض وطلب التماس إعادةالنظر.
.الدعاوى والطلبات التي يختص بها ديوانالمظالم.
.الدعاوى والطلبات المتعلقة بدعاوى قسمة التركات؛ عدا طلب النقض وطلب التماس إعادةالنظر.
".الإنهاءات وما يتعلق بها منطلبات.

[Retrieved Article 2]
Law: نظام التكاليف القضائية
Article: المادة الثامنة
Text:
اسم النظام: نظام التكاليف القضائية
رقم المادة: المادة الثامنة
النص القانوني: المادة الثامنة
١.تُقرض على الطلبات الآتية تكاليف قضائية بمبلغ لا يزيد على ألف ريال وفقاً للمعايير والضوابط وقواعد التقدير التي تحددهااللائحة

## 8. Measure Top-3 context sufficiency

Before generation, check whether the gold legal article is present in the Top-3.

This separates:
- **upstream retrieval/reranking limitations**, and
- **generator behavior**.

A generator cannot correctly cite a gold article that was never included in its context.

In [ ]:
ARABIC_DIACRITICS = re.compile(
    r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED\u0640]"
)

def normalize_label(value):
    s = str(value or "").strip()
    s = ARABIC_DIACRITICS.sub("", s)

    # Final Stage 3 citation-matching correction:
    # Qwen may output "الائحة" while the KB/context uses "اللائحة".
    # Normalize this spelling variation before citation comparison.
    s = s.replace("الائحة", "اللائحة")

    s = s.replace("أ", "ا").replace("إ", "ا").replace("آ", "ا").replace("ٱ", "ا")
    s = s.replace("ى", "ي").replace("ؤ", "و").replace("ئ", "ي")

    # After hamza normalization, normalize the missing-lam form too:
    # "اللائحة" -> "اللايحة", while "الائحة" may become "الايحة".
    if s.startswith("الايحة"):
        s = "اللايحة" + s[len("الايحة"):]

    s = re.sub(r"[\s\-_–—:،,؛;.!؟?()\[\]{}]+", "", s)
    return s

def pair_key(law, article):
    return normalize_label(law) + "||" + normalize_label(article)

retrieved_pairs_map = {}

for query_id, group in top3_df.groupby("query_id"):
    pairs = [
        (str(r["law_name"]).strip(), str(r["article_number"]).strip())
        for _, r in group.sort_values("rerank_rank").iterrows()
    ]
    retrieved_pairs_map[str(query_id)] = pairs

context_df["retrieved_pairs"] = context_df["query_id"].map(retrieved_pairs_map)

context_df["retrieved_pair_keys"] = context_df["retrieved_pairs"].apply(
    lambda xs: [pair_key(law, article) for law, article in xs]
)

context_df["gold_key"] = context_df.apply(
    lambda r: pair_key(r["gold_law"], r["gold_article"]),
    axis=1,
)

context_df["context_has_gold"] = context_df.apply(
    lambda r: r["gold_key"] in r["retrieved_pair_keys"],
    axis=1,
)

print(
    "Gold article available in Top-3:",
    int(context_df["context_has_gold"].sum()),
    "/",
    len(context_df),
)
print("Top-3 gold coverage:", f"{context_df['context_has_gold'].mean():.2%}")

display(
    context_df[
        [
            "query_id",
            "query_type",
            "gold_law",
            "gold_article",
            "context_has_gold",
        ]
    ].head(10)
)

Gold article available in Top-3: 54 / 60
Top-3 gold coverage: 90.00%


,query_id,query_type,gold_law,gold_article,context_has_gold
0,Q001,formal_direct,نظام التكاليف القضائية,المادة الثانية,True
1,Q002,formal_paraphrase,نظام التكاليف القضائية,المادة الثانية,False
2,Q003,informal,نظام التكاليف القضائية,المادة الثانية,False
3,Q004,formal_direct,نظام التكاليف القضائية,المادة الثالثة,True
4,Q005,formal_paraphrase,نظام التكاليف القضائية,المادة الثالثة,True
5,Q006,informal,نظام التكاليف القضائية,المادة الثالثة,True
6,Q007,formal_direct,نظام التكاليف القضائية,المادة الرابعة,True
7,Q008,formal_paraphrase,نظام التكاليف القضائية,المادة الرابعة,True
8,Q009,informal,نظام التكاليف القضائية,المادة الرابعة,True
9,Q010,formal_direct,نظام التكاليف القضائية,المادة الخامسة,True


## 9. Recover gold legal article text for evaluation only

The gold text is used only by the evaluation step.  
It is **never added to the generation prompt unless it was actually retrieved in Top-3**.

In [ ]:
kb_df["_law_norm"] = kb_df["law_name"].map(normalize_label)
kb_df["_article_norm"] = kb_df["article_number"].map(normalize_label)
kb_df["_gold_key"] = kb_df["_law_norm"] + "||" + kb_df["_article_norm"]

gold_text_map = (
    kb_df.drop_duplicates("_gold_key")
    .set_index("_gold_key")["text"]
    .astype(str)
    .to_dict()
)

context_df["gold_text"] = context_df["gold_key"].map(gold_text_map)

missing_gold_text = int(context_df["gold_text"].isna().sum())
print("Missing gold article texts:", missing_gold_text)

assert missing_gold_text == 0, (
    "Some gold articles could not be matched in the cleaned knowledge base."
)

print("✅ Gold reference texts recovered for evaluation only.")

Missing gold article texts: 0
✅ Gold reference texts recovered for evaluation only.


## 10. Build the shared augmented-generation prompt

Both generators receive exactly the same:
- user question
- Top-3 legal context
- instructions

The model must answer only from the retrieved context and return structured JSON.

In [ ]:
def build_generation_prompt(row):
    query = str(row["query"]).strip()
    context = str(row["augmented_context"]).strip()

    return f"""
أنت مساعد قانوني سعودي ضمن نظام SANAD.

أجب عن سؤال المستخدم اعتماداً فقط على السياق القانوني المسترجع أدناه.

قواعد إلزامية:
1) لا تستخدم أي معلومة قانونية من خارج السياق المسترجع.
2) لا تخترع نظاماً أو مادة أو رقماً غير موجود في السياق.
3) اذكر اسم النظام ورقم المادة كما ظهرا في السياق.
4) إذا كان السياق لا يكفي للإجابة بثقة، صرّح بذلك واجعل insufficient_context = true.
5) اجعل الإجابة باللغة العربية، واضحة، مختصرة، ومهنية.
6) لا تكتب أي نص خارج JSON.
7) أعد JSON صالحاً بهذا الشكل فقط:
{{
  "answer_ar": "الإجابة العربية",
  "citations": [
    {{"law": "اسم النظام", "article": "رقم المادة"}}
  ],
  "insufficient_context": false
}}

سؤال المستخدم:
{query}

السياق القانوني المسترجع:
{context}
""".strip()

context_df["generation_prompt"] = context_df.apply(
    build_generation_prompt,
    axis=1,
)

print(context_df.loc[0, "generation_prompt"][:3000])

أنت مساعد قانوني سعودي ضمن نظام SANAD.

أجب عن سؤال المستخدم اعتماداً فقط على السياق القانوني المسترجع أدناه.

قواعد إلزامية:
1) لا تستخدم أي معلومة قانونية من خارج السياق المسترجع.
2) لا تخترع نظاماً أو مادة أو رقماً غير موجود في السياق.
3) اذكر اسم النظام ورقم المادة كما ظهرا في السياق.
4) إذا كان السياق لا يكفي للإجابة بثقة، صرّح بذلك واجعل insufficient_context = true.
5) اجعل الإجابة باللغة العربية، واضحة، مختصرة، ومهنية.
6) لا تكتب أي نص خارج JSON.
7) أعد JSON صالحاً بهذا الشكل فقط:
{
  "answer_ar": "الإجابة العربية",
  "citations": [
    {"law": "اسم النظام", "article": "رقم المادة"}
  ],
  "insufficient_context": false
}

سؤال المستخدم:
ما الدعاوى والطلبات التي لا تسري عليها أحكام التكاليف القضائية؟

السياق القانوني المسترجع:
[Retrieved Article 1]
Law: نظام التكاليف القضائية
Article: المادة الثانية
Text:
اسم النظام: نظام التكاليف القضائية
رقم المادة: المادة الثانية
النص القانوني: المادة الثانية
تسري أحكام النظام على جميع الدعاوى والطلبات التي تقدم إلى المحاكم» فيما عدا مايأتي:
١

## 11. Generator configuration

The two models are loaded sequentially to reduce GPU memory usage.

The same input and output token limits and deterministic decoding are used for both models.

In [ ]:
ALLAM_MODEL_ID = "humain-ai/ALLaM-7B-Instruct-preview"
QWEN_MODEL_ID = "Qwen/Qwen3-8B"

MAX_INPUT_TOKENS = 3400
MAX_NEW_TOKENS = 350

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

def load_generator(model_id, revision=None):
    print("Loading:", model_id)

    tokenizer = AutoTokenizer.from_pretrained(
        model_id,
        revision=revision,
        use_fast=True,
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        revision=revision,
        quantization_config=bnb_config,
        device_map="auto",
        low_cpu_mem_usage=True,
    )
    model.eval()

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    print("✅ Loaded:", model_id)
    return tokenizer, model

def release_model(tokenizer=None, model=None):
    if model is not None:
        del model
    if tokenizer is not None:
        del tokenizer

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def format_for_model(tokenizer, model_id, prompt):
    messages = [{"role": "user", "content": prompt}]

    kwargs = {
        "tokenize": False,
        "add_generation_prompt": True,
    }

    # Qwen3 thinking is disabled for a fair direct-answer comparison.
    if model_id == QWEN_MODEL_ID:
        kwargs["enable_thinking"] = False

    try:
        return tokenizer.apply_chat_template(messages, **kwargs)
    except TypeError:
        kwargs.pop("enable_thinking", None)
        return tokenizer.apply_chat_template(messages, **kwargs)
    except Exception:
        return prompt

def generate_one(
    tokenizer,
    model,
    model_id,
    prompt,
    max_new_tokens=MAX_NEW_TOKENS,
):
    formatted = format_for_model(tokenizer, model_id, prompt)

    inputs = tokenizer(
        formatted,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_TOKENS,
    )

    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    input_tokens = int(inputs["input_ids"].shape[-1])
    input_truncated = input_tokens >= MAX_INPUT_TOKENS

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    start = time.perf_counter()

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    elapsed = time.perf_counter() - start

    new_tokens = output_ids[0][inputs["input_ids"].shape[-1]:]
    response = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True,
    ).strip()

    return response, elapsed, input_tokens, input_truncated

## 12. Resumable generation helper

Each generated answer is saved immediately.  
If Colab disconnects, rerunning the notebook resumes from unfinished query IDs.

In [ ]:

def run_generator(
    model_label,
    model_id,
    context_df,
    checkpoint_path,
    revision=None,
):
    checkpoint_path = Path(checkpoint_path)

    expected_ids = set(context_df["query_id"].astype(str))

    if checkpoint_path.exists():
        previous = pd.read_csv(
            checkpoint_path,
            dtype={"query_id": str}
        )
        results = previous.to_dict("records")
        completed_ids = set(previous["query_id"].astype(str))

        print(
            f"Resuming {model_label}: "
            f"{len(completed_ids)}/{len(context_df)} completed."
        )
    else:
        results = []
        completed_ids = set()

    # Skip model loading if all questions are already completed
    if expected_ids.issubset(completed_ids):
        print("✅ All questions already completed. Skipping model loading.")

        result_df = pd.DataFrame(results)
        result_df["query_id"] = result_df["query_id"].astype(str)

        result_df = result_df.drop_duplicates(
            subset=["query_id", "model"],
            keep="last",
        )

        assert result_df["query_id"].nunique() == len(context_df)

        return result_df

    # Load model only if unfinished questions remain
    tokenizer, model = load_generator(model_id, revision=revision)

    try:
        for _, row in tqdm(
            context_df.iterrows(),
            total=len(context_df),
            desc=f"{model_label} generation",
        ):
            qid = str(row["query_id"])

            if qid in completed_ids:
                continue

            response, elapsed, input_tokens, was_truncated = generate_one(
                tokenizer,
                model,
                model_id,
                row["generation_prompt"],
            )

            results.append({
                "query_id": qid,
                "model": model_label,
                "raw_response": response,
                "generation_time_s": elapsed,
                "input_tokens": input_tokens,
                "input_truncated": was_truncated,
            })

            pd.DataFrame(results).to_csv(
                checkpoint_path,
                index=False,
                encoding="utf-8-sig",
            )

            completed_ids.add(qid)

    finally:
        release_model(tokenizer, model)

    result_df = pd.DataFrame(results)
    result_df["query_id"] = result_df["query_id"].astype(str)

    result_df = result_df.drop_duplicates(
        subset=["query_id", "model"],
        keep="last",
    )

    result_df.to_csv(
        checkpoint_path,
        index=False,
        encoding="utf-8-sig",
    )

    assert result_df["query_id"].nunique() == len(context_df), (
        f"{model_label} did not generate all queries."
    )

    return result_df


## 13. Optional smoke test

In [ ]:
RUN_SMOKE_TEST = False

if RUN_SMOKE_TEST:
    tok, mdl = load_generator(ALLAM_MODEL_ID)

    response, elapsed, tokens, truncated = generate_one(
        tok,
        mdl,
        ALLAM_MODEL_ID,
        context_df.loc[0, "generation_prompt"],
        max_new_tokens=180,
    )

    print("Response:")
    print(response)
    print("Time:", elapsed)
    print("Input tokens:", tokens)
    print("Input truncated:", truncated)

    release_model(tok, mdl)

## 14. Generate answers with ALLaM

In [ ]:
ALLAM_CHECKPOINT = WORK_DIR / "allam_generation_checkpoint.csv"

allam_df = run_generator(
    model_label="ALLaM-7B-Instruct-preview",
    model_id=ALLAM_MODEL_ID,
    context_df=context_df,
    checkpoint_path=ALLAM_CHECKPOINT,
)

display(allam_df.head())

Resuming ALLaM-7B-Instruct-preview: 60/60 completed.
✅ All questions already completed. Skipping model loading.


,query_id,model,raw_response,generation_time_s,input_tokens,input_truncated
0,Q001,ALLaM-7B-Instruct-preview,"{\n ""answer_ar"": ""الدعاوى الجزائية العامة، ال...",9.581779,720,False
1,Q002,ALLaM-7B-Instruct-preview,"{\n ""answer_ar"": ""الحالات المستثناة من تطبيق ...",8.866128,728,False
2,Q003,ALLaM-7B-Instruct-preview,"{\n ""answer_ar"": ""القضايا التي لا تفرض عليها ...",11.579876,786,False
3,Q004,ALLaM-7B-Instruct-preview,"{\n ""answer_ar"": ""الحد الأعلى للتكاليف القضائ...",5.514837,571,False
4,Q005,ALLaM-7B-Instruct-preview,"{\n ""answer_ar"": ""تُقدّر التكاليف القضائية لل...",17.840053,798,False


## 15. Generate answers with Qwen3-8B

In [ ]:
QWEN_CHECKPOINT = WORK_DIR / "qwen3_generation_checkpoint.csv"

qwen_df = run_generator(
    model_label="Qwen3-8B",
    model_id=QWEN_MODEL_ID,
    context_df=context_df,
    checkpoint_path=QWEN_CHECKPOINT,
)

display(qwen_df.head())

Resuming Qwen3-8B: 60/60 completed.
✅ All questions already completed. Skipping model loading.


,query_id,model,raw_response,generation_time_s,input_tokens,input_truncated
0,Q001,Qwen3-8B,"{\n ""answer_ar"": ""الدعاوى والطلبات التي لا تس...",20.618407,973,False
1,Q002,Qwen3-8B,"{\n ""answer_ar"": ""الحالات المستثناة من تطبيق ...",14.981097,988,False
2,Q003,Qwen3-8B,"{\n ""answer_ar"": ""القضايا التي لا تُفرض عليها...",14.826432,1065,False
3,Q004,Qwen3-8B,"{\n ""answer_ar"": ""الحد الأعلى للتكاليف القضائ...",9.987638,754,False
4,Q005,Qwen3-8B,"{\n ""answer_ar"": ""تُحدد قيمة التكاليف القضائي...",21.489913,1075,False


## 16. Parse generator JSON outputs

The parser accepts:
- valid JSON
- JSON inside Markdown code fences

Invalid output is preserved and marked with `valid_json = 0`.

In [ ]:
def extract_json_object(text):
    s = str(text or "").strip()

    # Remove Markdown code fences if present.
    s = re.sub(r"^```(?:json)?\s*", "", s, flags=re.IGNORECASE)
    s = re.sub(r"\s*```$", "", s)

    try:
        return json.loads(s), True
    except Exception:
        pass

    match = re.search(r"\{.*\}", s, flags=re.DOTALL)

    if match:
        candidate = match.group(0)

        try:
            return json.loads(candidate), True
        except Exception:
            pass

    return {}, False

def normalize_citations(value):
    if not isinstance(value, list):
        return []

    out = []

    for item in value:
        if not isinstance(item, dict):
            continue

        law = str(item.get("law", "")).strip()
        article = str(item.get("article", "")).strip()

        if law and article:
            out.append({
                "law": law,
                "article": article,
            })

    return out

def parse_generation_df(df):
    rows = []

    for _, row in df.iterrows():
        obj, valid = extract_json_object(row["raw_response"])

        rows.append({
            **row.to_dict(),
            "valid_json": int(valid),
            "answer_ar": str(obj.get("answer_ar", "")).strip(),
            "citations": normalize_citations(obj.get("citations", [])),
            "insufficient_context": bool(
                obj.get("insufficient_context", False)
            ),
        })

    return pd.DataFrame(rows)

allam_parsed = parse_generation_df(allam_df)
qwen_parsed = parse_generation_df(qwen_df)

generation_df = pd.concat(
    [allam_parsed, qwen_parsed],
    ignore_index=True,
)

print("Generation rows:", len(generation_df))
print(
    generation_df.groupby("model")["valid_json"].mean()
    .rename("Valid JSON rate")
)

Generation rows: 120
model
ALLaM-7B-Instruct-preview    0.983333
Qwen3-8B                     1.000000
Name: Valid JSON rate, dtype: float64


## 17. Merge generation outputs with the fixed Stage 3 context

In [ ]:
base_eval_columns = [
    "query_id",
    "query_type",
    "query",
    "gold_law",
    "gold_article",
    "gold_key",
    "gold_text",
    "selected_retrieval",
    "selected_reranker",
    "augmented_context",
    "retrieved_pairs",
    "retrieved_pair_keys",
    "context_has_gold",
]

eval_df = generation_df.merge(
    context_df[base_eval_columns],
    on="query_id",
    how="left",
    validate="many_to_one",
)

assert len(eval_df) == 120, (
    f"Expected 120 model-query rows, found {len(eval_df)}."
)

print("Evaluation rows:", len(eval_df))
display(eval_df.head())

Evaluation rows: 120


,query_id,model,raw_response,generation_time_s,input_tokens,input_truncated,valid_json,answer_ar,citations,insufficient_context,...,gold_law,gold_article,gold_key,gold_text,selected_retrieval,selected_reranker,augmented_context,retrieved_pairs,retrieved_pair_keys,context_has_gold
0,Q001,ALLaM-7B-Instruct-preview,"{\n ""answer_ar"": ""الدعاوى الجزائية العامة، ال...",9.581779,720,False,1,الدعاوى الجزائية العامة، الدعاوى التأديبية، ال...,"[{'law': 'نظام التكاليف القضائية', 'article': ...",False,...,نظام التكاليف القضائية,المادة الثانية,نظامالتكاليفالقضايية||المادةالثانية,المادة الثانية\nتسري أحكام النظام على جميع الد...,Dense,BAAI/bge-reranker-v2-m3,[Retrieved Article 1]\nLaw: نظام التكاليف القض...,"[(نظام التكاليف القضائية, المادة الثانية), (نظ...","[نظامالتكاليفالقضايية||المادةالثانية, نظامالتك...",True
1,Q002,ALLaM-7B-Instruct-preview,"{\n ""answer_ar"": ""الحالات المستثناة من تطبيق ...",8.866128,728,False,1,الحالات المستثناة من تطبيق نظام التكاليف القضا...,"[{'law': 'نظام التكاليف القضائية', 'article': ...",False,...,نظام التكاليف القضائية,المادة الثانية,نظامالتكاليفالقضايية||المادةالثانية,المادة الثانية\nتسري أحكام النظام على جميع الد...,Dense,BAAI/bge-reranker-v2-m3,[Retrieved Article 1]\nLaw: نظام التكاليف القض...,"[(نظام التكاليف القضائية, المادة السابعةعشرة),...","[نظامالتكاليفالقضايية||المادةالسابعةعشرة, نظام...",False
2,Q003,ALLaM-7B-Instruct-preview,"{\n ""answer_ar"": ""القضايا التي لا تفرض عليها ...",11.579876,786,False,1,القضايا التي لا تفرض عليها تكاليف قضائية تشمل ...,"[{'law': 'نظام التكاليف القضائية', 'article': ...",False,...,نظام التكاليف القضائية,المادة الثانية,نظامالتكاليفالقضايية||المادةالثانية,المادة الثانية\nتسري أحكام النظام على جميع الد...,Dense,BAAI/bge-reranker-v2-m3,[Retrieved Article 1]\nLaw: نظام التكاليف القض...,"[(نظام التكاليف القضائية, المادة السابعةعشرة),...","[نظامالتكاليفالقضايية||المادةالسابعةعشرة, اللا...",False
3,Q004,ALLaM-7B-Instruct-preview,"{\n ""answer_ar"": ""الحد الأعلى للتكاليف القضائ...",5.514837,571,False,1,الحد الأعلى للتكاليف القضائية المفروضة على الد...,"[{'law': 'نظام التكاليف القضائية', 'article': ...",False,...,نظام التكاليف القضائية,المادة الثالثة,نظامالتكاليفالقضايية||المادةالثالثة,المادة الثالثة\nتفرض تكاليف قضائية على الدعوى ...,Dense,BAAI/bge-reranker-v2-m3,[Retrieved Article 1]\nLaw: نظام التكاليف القض...,"[(نظام التكاليف القضائية, المادة الثالثة), (نظ...","[نظامالتكاليفالقضايية||المادةالثالثة, نظامالتك...",True
4,Q005,ALLaM-7B-Instruct-preview,"{\n ""answer_ar"": ""تُقدّر التكاليف القضائية لل...",17.840053,798,False,1,تُقدّر التكاليف القضائية للدعوى المحددة القيمة...,[{'law': 'اللائحة التنفيذية لنظام التكاليف الق...,False,...,نظام التكاليف القضائية,المادة الثالثة,نظامالتكاليفالقضايية||المادةالثالثة,المادة الثالثة\nتفرض تكاليف قضائية على الدعوى ...,Dense,BAAI/bge-reranker-v2-m3,[Retrieved Article 1]\nLaw: اللائحة التنفيذية ...,"[(اللائحة التنفيذية لنظام التكاليف القضائية, ا...",[اللايحةالتنفيذيةلنظامالتكاليفالقضايية||المادة...,True


### Citation normalization used in this final version

Before comparing `(law, article)` pairs, the evaluator normalizes Arabic spelling and treats `الائحة` and `اللائحة` as the same intended legal label. This avoids penalizing a model for that spelling variation while still requiring the cited law/article pair to exist in the retrieved Top-3.


## 18. Deterministic citation verification

Citation verification does **not** use another LLM.

Metrics:
- Citation Coverage
- Citation Precision vs retrieved Top-3
- Gold Citation Hit
- Citation Hallucination Rate

A citation is considered grounded only if the cited `(law, article)` pair appears in the supplied Top-3.

In [ ]:
def citation_metrics(row):
    citations = row["citations"]

    cited_keys = [
        pair_key(c.get("law", ""), c.get("article", ""))
        for c in citations
    ]

    retrieved_keys = set(row["retrieved_pair_keys"])
    gold_key = row["gold_key"]

    total = len(cited_keys)
    valid_count = sum(key in retrieved_keys for key in cited_keys)

    return pd.Series({
        "citation_coverage": int(total > 0),
        "citation_precision_context": (
            valid_count / total if total else 0.0
        ),
        "gold_citation_hit": int(gold_key in cited_keys),
        "citation_hallucination_rate": (
            (total - valid_count) / total if total else 0.0
        ),
        "citation_count": total,
    })

citation_eval = eval_df.apply(citation_metrics, axis=1)
eval_df = pd.concat([eval_df, citation_eval], axis=1)

display(
    eval_df[
        [
            "query_id",
            "model",
            "context_has_gold",
            "citation_coverage",
            "citation_precision_context",
            "gold_citation_hit",
            "citation_hallucination_rate",
        ]
    ].head(12)
)

,query_id,model,context_has_gold,citation_coverage,citation_precision_context,gold_citation_hit,citation_hallucination_rate
0,Q001,ALLaM-7B-Instruct-preview,True,1.0,1.0,1.0,0.0
1,Q002,ALLaM-7B-Instruct-preview,False,1.0,1.0,0.0,0.0
2,Q003,ALLaM-7B-Instruct-preview,False,1.0,1.0,0.0,0.0
3,Q004,ALLaM-7B-Instruct-preview,True,1.0,1.0,1.0,0.0
4,Q005,ALLaM-7B-Instruct-preview,True,1.0,1.0,0.0,0.0
5,Q006,ALLaM-7B-Instruct-preview,True,1.0,1.0,1.0,0.0
6,Q007,ALLaM-7B-Instruct-preview,True,1.0,1.0,1.0,0.0
7,Q008,ALLaM-7B-Instruct-preview,True,1.0,1.0,1.0,0.0
8,Q009,ALLaM-7B-Instruct-preview,True,1.0,1.0,1.0,0.0
9,Q010,ALLaM-7B-Instruct-preview,True,1.0,1.0,1.0,0.0


## 19. Automatic citation and generation summary

In [ ]:
automatic_rows = []

for model_name, g in eval_df.groupby("model"):
    conditional = g[g["context_has_gold"] == True]

    automatic_rows.append({
        "Model": model_name,
        "N": len(g),
        "Valid_JSON_Rate": g["valid_json"].mean(),
        "Input_Truncation_Rate": g["input_truncated"].mean(),
        "Citation_Coverage": g["citation_coverage"].mean(),
        "Citation_Precision_Context": g["citation_precision_context"].mean(),
        "Gold_Citation_Accuracy_All": g["gold_citation_hit"].mean(),
        "Gold_Citation_Accuracy_ContextHasGold": (
            conditional["gold_citation_hit"].mean()
            if len(conditional) else np.nan
        ),
        "Citation_Hallucination_Rate": g[
            "citation_hallucination_rate"
        ].mean(),
        "Insufficient_Context_Rate": g[
            "insufficient_context"
        ].mean(),
        "Avg_Generation_Time_s": g["generation_time_s"].mean(),
    })

automatic_summary_df = pd.DataFrame(automatic_rows)
display(automatic_summary_df)

,Model,N,Valid_JSON_Rate,Input_Truncation_Rate,Citation_Coverage,Citation_Precision_Context,Gold_Citation_Accuracy_All,Gold_Citation_Accuracy_ContextHasGold,Citation_Hallucination_Rate,Insufficient_Context_Rate,Avg_Generation_Time_s
0,ALLaM-7B-Instruct-preview,60,0.983333,0.0,0.983333,0.958333,0.800000,0.888889,0.025000,0.0,9.038947
1,Qwen3-8B,60,1.000000,0.0,1.000000,0.955556,0.766667,0.851852,0.044444,0.0,16.028851


## 20. Independent content-quality evaluation

Citation checks verify references, but they do not fully measure answer correctness, faithfulness, or Arabic quality.

A separate judge model evaluates:
- Correctness (1–5)
- Faithfulness (1–5)
- Arabic quality (1–5)
- Hallucination flag (0/1)

The judge receives:
- the question
- gold legal article text
- retrieved Top-3 context
- generated answer

It is instructed not to use outside legal knowledge.

> The LLM judge is supplemental evidence. A blinded human-review file is also exported later.

In [ ]:
RUN_LLM_JUDGE = True

# Public judge model used after the two generators are released from GPU.
JUDGE_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
JUDGE_MAX_NEW_TOKENS = 220

def build_judge_prompt(row):
    return f"""
أنت مقيم مستقل لإجابات نظام قانوني سعودي قائم على RAG.

قيّم الإجابة فقط باستخدام:
1) سؤال المستخدم،
2) النص القانوني المرجعي الذهبي،
3) السياق القانوني المسترجع Top-3.

لا تستخدم معرفة قانونية خارجية.

أعد JSON صالحاً فقط بهذا الشكل:
{{
  "correctness": 1,
  "faithfulness": 1,
  "arabic_quality": 1,
  "hallucination": 0,
  "reason": "سبب مختصر"
}}

التقييم:
- correctness من 1 إلى 5: مدى صحة الإجابة بالنسبة للسؤال والنص المرجعي.
- faithfulness من 1 إلى 5: مدى دعم السياق المسترجع لادعاءات الإجابة.
- arabic_quality من 1 إلى 5: وضوح وسلامة ومهنية العربية.
- hallucination = 1 إذا احتوت الإجابة على ادعاء قانوني مادي غير مدعوم بالسياق المسترجع، وإلا 0.

سؤال المستخدم:
{row['query']}

النص القانوني المرجعي الذهبي:
{row['gold_text']}

السياق المسترجع Top-3:
{row['augmented_context']}

الإجابة المراد تقييمها:
{row['answer_ar']}
""".strip()

## 21. Run the independent judge with checkpointing

In [ ]:
def parse_judge_response(text):
    obj, valid = extract_json_object(text)

    def safe_score(name):
        try:
            value = float(obj.get(name))
            return min(5.0, max(1.0, value))
        except Exception:
            return np.nan

    try:
        hallucination = int(obj.get("hallucination"))
        hallucination = 1 if hallucination else 0
    except Exception:
        hallucination = np.nan

    return {
        "judge_valid_json": int(valid),
        "correctness": safe_score("correctness"),
        "faithfulness": safe_score("faithfulness"),
        "arabic_quality": safe_score("arabic_quality"),
        "hallucination": hallucination,
        "judge_reason": str(obj.get("reason", "")).strip(),
    }

def run_judge(eval_df, checkpoint_path):
    checkpoint_path = Path(checkpoint_path)

    if checkpoint_path.exists():
        previous = pd.read_csv(
            checkpoint_path,
            dtype={"query_id": str, "model": str},
        )
        results = previous.to_dict("records")
        completed = set(
            zip(
                previous["query_id"].astype(str),
                previous["model"].astype(str),
            )
        )
        print(f"Resuming judge: {len(completed)}/{len(eval_df)} completed.")
    else:
        results = []
        completed = set()

    tokenizer, model = load_generator(JUDGE_MODEL_ID)

    try:
        for _, row in tqdm(
            eval_df.iterrows(),
            total=len(eval_df),
            desc="Independent judge",
        ):
            key = (str(row["query_id"]), str(row["model"]))

            if key in completed:
                continue

            judge_prompt = build_judge_prompt(row)

            response, elapsed, input_tokens, was_truncated = generate_one(
                tokenizer,
                model,
                JUDGE_MODEL_ID,
                judge_prompt,
                max_new_tokens=JUDGE_MAX_NEW_TOKENS,
            )

            parsed = parse_judge_response(response)

            results.append({
                "query_id": key[0],
                "model": key[1],
                "judge_raw_response": response,
                "judge_time_s": elapsed,
                "judge_input_tokens": input_tokens,
                "judge_input_truncated": was_truncated,
                **parsed,
            })

            pd.DataFrame(results).to_csv(
                checkpoint_path,
                index=False,
                encoding="utf-8-sig",
            )

            completed.add(key)

    finally:
        release_model(tokenizer, model)

    judge_df = pd.DataFrame(results)

    judge_df["query_id"] = judge_df["query_id"].astype(str)
    judge_df["model"] = judge_df["model"].astype(str)

    judge_df = judge_df.drop_duplicates(
        subset=["query_id", "model"],
        keep="last",
    )

    judge_df.to_csv(
        checkpoint_path,
        index=False,
        encoding="utf-8-sig",
    )

    return judge_df


JUDGE_CHECKPOINT = WORK_DIR / "stage3_judge_checkpoint.csv"

if RUN_LLM_JUDGE:
    judge_df = run_judge(
        eval_df,
        JUDGE_CHECKPOINT,
    )

    old_judge_cols = [
        "judge_raw_response",
        "judge_time_s",
        "judge_input_tokens",
        "judge_input_truncated",
        "judge_valid_json",
        "correctness",
        "faithfulness",
        "arabic_quality",
        "hallucination",
        "judge_reason",
    ]

    eval_df = eval_df.drop(
        columns=[c for c in old_judge_cols if c in eval_df.columns],
        errors="ignore",
    )

    eval_df = eval_df.merge(
        judge_df,
        on=["query_id", "model"],
        how="left",
        validate="one_to_one",
    )

    print("✅ Judge rows:", len(judge_df))
    print(
        "Valid judge JSON:",
        f"{eval_df['judge_valid_json'].mean():.2%}",
    )
else:
    print("LLM judge skipped.")

Resuming judge: 120/120 completed.
Loading: Qwen/Qwen2.5-7B-Instruct


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (4 outputs left out)

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

✅ Loaded: Qwen/Qwen2.5-7B-Instruct


Independent judge:   0%|          | 0/120 [00:00<?, ?it/s]

✅ Judge rows: 120
Valid judge JSON: 100.00%


## 22. Final generator comparison

Primary selection dimensions:
1. Correctness
2. Faithfulness
3. Citation accuracy
4. Lower hallucination
5. Arabic quality

Generation time is a secondary efficiency metric.

In [ ]:
final_rows = []

for model_name, g in eval_df.groupby("model"):
    conditional = g[g["context_has_gold"] == True]

    final_rows.append({
        "Model": model_name,
        "Correctness_1_5": (
            g["correctness"].mean()
            if "correctness" in g.columns else np.nan
        ),
        "Faithfulness_1_5": (
            g["faithfulness"].mean()
            if "faithfulness" in g.columns else np.nan
        ),
        "Citation_Accuracy_All": g["gold_citation_hit"].mean(),
        "Citation_Accuracy_ContextHasGold": (
            conditional["gold_citation_hit"].mean()
            if len(conditional) else np.nan
        ),
        "Citation_Precision_Context": g[
            "citation_precision_context"
        ].mean(),
        "Hallucination_Rate": (
            g["hallucination"].mean()
            if "hallucination" in g.columns else np.nan
        ),
        "Citation_Hallucination_Rate": g[
            "citation_hallucination_rate"
        ].mean(),
        "Arabic_Quality_1_5": (
            g["arabic_quality"].mean()
            if "arabic_quality" in g.columns else np.nan
        ),
        "Avg_Generation_Time_s": g["generation_time_s"].mean(),
        "Valid_JSON_Rate": g["valid_json"].mean(),
        "Input_Truncation_Rate": g["input_truncated"].mean(),
    })

final_comparison_df = pd.DataFrame(final_rows)
display(final_comparison_df)

,Model,Correctness_1_5,Faithfulness_1_5,Citation_Accuracy_All,Citation_Accuracy_ContextHasGold,Citation_Precision_Context,Hallucination_Rate,Citation_Hallucination_Rate,Arabic_Quality_1_5,Avg_Generation_Time_s,Valid_JSON_Rate,Input_Truncation_Rate
0,ALLaM-7B-Instruct-preview,4.416667,4.150000,0.800000,0.888889,0.958333,0.066667,0.025000,4.666667,9.038947,0.983333,0.0
1,Qwen3-8B,4.316667,3.966667,0.766667,0.851852,0.955556,0.050000,0.044444,4.566667,16.028851,1.000000,0.0


## 23. Results by query type

In [ ]:
by_type_rows = []

for (query_type, model_name), g in eval_df.groupby(
    ["query_type", "model"]
):
    conditional = g[g["context_has_gold"] == True]

    by_type_rows.append({
        "Query_Type": query_type,
        "Model": model_name,
        "N": len(g),
        "Correctness_1_5": (
            g["correctness"].mean()
            if "correctness" in g.columns else np.nan
        ),
        "Faithfulness_1_5": (
            g["faithfulness"].mean()
            if "faithfulness" in g.columns else np.nan
        ),
        "Citation_Accuracy_All": g["gold_citation_hit"].mean(),
        "Citation_Accuracy_ContextHasGold": (
            conditional["gold_citation_hit"].mean()
            if len(conditional) else np.nan
        ),
        "Hallucination_Rate": (
            g["hallucination"].mean()
            if "hallucination" in g.columns else np.nan
        ),
        "Arabic_Quality_1_5": (
            g["arabic_quality"].mean()
            if "arabic_quality" in g.columns else np.nan
        ),
        "Avg_Generation_Time_s": g["generation_time_s"].mean(),
    })

by_query_type_df = pd.DataFrame(by_type_rows)
display(by_query_type_df)

,Query_Type,Model,N,Correctness_1_5,Faithfulness_1_5,Citation_Accuracy_All,Citation_Accuracy_ContextHasGold,Hallucination_Rate,Arabic_Quality_1_5,Avg_Generation_Time_s
0,formal_direct,ALLaM-7B-Instruct-preview,20,4.75,4.60,0.90,0.900000,0.00,4.90,8.760202
1,formal_direct,Qwen3-8B,20,4.50,4.15,0.90,0.900000,0.00,4.65,16.697247
2,formal_paraphrase,ALLaM-7B-Instruct-preview,20,4.40,4.05,0.75,0.882353,0.10,4.65,9.702427
3,formal_paraphrase,Qwen3-8B,20,4.10,3.75,0.65,0.764706,0.10,4.50,17.162832
4,informal,ALLaM-7B-Instruct-preview,20,4.10,3.80,0.75,0.882353,0.10,4.45,8.654211
5,informal,Qwen3-8B,20,4.35,4.00,0.75,0.882353,0.05,4.55,14.226474


## 24. Paired bootstrap 95% confidence intervals

Both models answer the same 60 questions, so comparisons are paired.

If a confidence interval includes zero, do not claim a decisive difference for that metric.

In [ ]:
rng = np.random.default_rng(42)
N_BOOT = 5000

def paired_bootstrap_difference(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)

    mask = ~(np.isnan(a) | np.isnan(b))
    a = a[mask]
    b = b[mask]

    if len(a) == 0:
        return np.nan, np.nan, np.nan

    observed = b.mean() - a.mean()
    diffs = np.empty(N_BOOT, dtype=float)
    n = len(a)

    for i in range(N_BOOT):
        idx = rng.integers(0, n, size=n)
        diffs[i] = b[idx].mean() - a[idx].mean()

    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return observed, lo, hi

wide = eval_df.pivot(index="query_id", columns="model")

MODEL_A = "ALLaM-7B-Instruct-preview"
MODEL_B = "Qwen3-8B"

metrics_for_ci = [
    ("Correctness", "correctness"),
    ("Faithfulness", "faithfulness"),
    ("Gold Citation Hit", "gold_citation_hit"),
    ("Citation Precision", "citation_precision_context"),
    ("Arabic Quality", "arabic_quality"),
]

ci_rows = []

for label, col in metrics_for_ci:
    if col not in eval_df.columns:
        continue

    a = wide[col][MODEL_A]
    b = wide[col][MODEL_B]

    diff, lo, hi = paired_bootstrap_difference(a, b)

    ci_rows.append({
        "Metric": label,
        "Difference_Qwen_minus_ALLaM": diff,
        "95%_CI_Lower": lo,
        "95%_CI_Upper": hi,
    })

bootstrap_ci_df = pd.DataFrame(ci_rows)
display(bootstrap_ci_df)

,Metric,Difference_Qwen_minus_ALLaM,95%_CI_Lower,95%_CI_Upper
0,Correctness,-0.100000,-0.300000,0.133333
1,Faithfulness,-0.183333,-0.466667,0.100000
2,Gold Citation Hit,-0.033333,-0.116667,0.050000
3,Citation Precision,-0.002778,-0.061111,0.058333
4,Arabic Quality,-0.100000,-0.266667,0.100000


## 25. Select the final generator from the actual results

The selection is not hard-coded.

Priority:
1. Higher correctness
2. Higher faithfulness
3. Higher citation accuracy
4. Lower hallucination
5. Higher Arabic quality
6. Lower generation time

In [ ]:
selection_df = final_comparison_df.copy()

# If the LLM judge was skipped, quality columns may be NaN.
# Fill only for sorting so the automatic selection remains executable.
selection_df["_correctness"] = selection_df["Correctness_1_5"].fillna(-1)
selection_df["_faithfulness"] = selection_df["Faithfulness_1_5"].fillna(-1)
selection_df["_hallucination"] = selection_df["Hallucination_Rate"].fillna(1)
selection_df["_arabic"] = selection_df["Arabic_Quality_1_5"].fillna(-1)

selection_df = selection_df.sort_values(
    by=[
        "_correctness",
        "_faithfulness",
        "Citation_Accuracy_All",
        "_hallucination",
        "_arabic",
        "Avg_Generation_Time_s",
    ],
    ascending=[False, False, False, True, False, True],
    kind="stable",
).reset_index(drop=True)

SELECTED_GENERATOR = str(selection_df.loc[0, "Model"])

print("Selected Stage 3 generator:", SELECTED_GENERATOR)
display(
    selection_df[
        [
            "Model",
            "Correctness_1_5",
            "Faithfulness_1_5",
            "Citation_Accuracy_All",
            "Hallucination_Rate",
            "Arabic_Quality_1_5",
            "Avg_Generation_Time_s",
        ]
    ]
)

Selected Stage 3 generator: ALLaM-7B-Instruct-preview


,Model,Correctness_1_5,Faithfulness_1_5,Citation_Accuracy_All,Hallucination_Rate,Arabic_Quality_1_5,Avg_Generation_Time_s
0,ALLaM-7B-Instruct-preview,4.416667,4.150000,0.800000,0.066667,4.666667,9.038947
1,Qwen3-8B,4.316667,3.966667,0.766667,0.050000,4.566667,16.028851


## 26. Blinded human-review template

The LLM judge is supplemental evidence, not absolute ground truth.

This file allows reviewers to manually score the two model answers without seeing model identities.

In [ ]:
human_review = eval_df[
    [
        "query_id",
        "query_type",
        "query",
        "gold_law",
        "gold_article",
        "gold_text",
        "augmented_context",
        "model",
        "answer_ar",
    ]
].copy()

blind_map = {
    "ALLaM-7B-Instruct-preview": "Model A",
    "Qwen3-8B": "Model B",
}

human_review["blind_model"] = human_review["model"].map(blind_map)
human_review = human_review.drop(columns=["model"])

human_review["Human_Correctness_1_5"] = ""
human_review["Human_Faithfulness_1_5"] = ""
human_review["Human_Arabic_Quality_1_5"] = ""
human_review["Human_Hallucination_0_1"] = ""
human_review["Human_Notes"] = ""

human_review = human_review.sample(
    frac=1,
    random_state=SEED,
).reset_index(drop=True)

HUMAN_REVIEW_PATH = WORK_DIR / "SANAD_Stage3_Human_Review_Blinded.csv"

human_review.to_csv(
    HUMAN_REVIEW_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("Saved:", HUMAN_REVIEW_PATH)
display(human_review.head())

Saved: /content/drive/MyDrive/SANAD_Stage3_REBUILT/SANAD_Stage3_Human_Review_Blinded.csv


,query_id,query_type,query,gold_law,gold_article,gold_text,augmented_context,answer_ar,blind_model,Human_Correctness_1_5,Human_Faithfulness_1_5,Human_Arabic_Quality_1_5,Human_Hallucination_0_1,Human_Notes
0,Q045,informal,أقدر أرافع بنفسي قدام الدائرة التجارية بالمحكم...,اللائحة التنفيذية لنظام المحاكم التجارية,المادة الرابعة والخمسون,المادة الرابعة والخمسون : الترافع منمحام\nفي ج...,[Retrieved Article 1]\nLaw: اللائحة التنفيذية ...,يجب أن تكون المرافعة وتقديم أي مذكرات لدى الدا...,Model A,,,,,
1,Q048,informal,إذا تصالحنا بعد ما سجلنا القضية، وش يصير؟,اللائحة التنفيذية لنظام المحاكم التجارية,المادة الحادية والستون,المادة الحادية والستون : المصالحة بعدالقيد\nإذ...,[Retrieved Article 1]\nLaw: اللائحة التنفيذية ...,إذا تصالح الأطراف بعد قيد القضية، يُثبت ما اتف...,Model A,,,,,
2,Q005,formal_paraphrase,كيف يتم تحديد قيمة التكاليف القضائية بالنسبة إ...,نظام التكاليف القضائية,المادة الثالثة,المادة الثالثة\nتفرض تكاليف قضائية على الدعوى ...,[Retrieved Article 1]\nLaw: اللائحة التنفيذية ...,تُقدّر التكاليف القضائية للدعوى المحددة القيمة...,Model A,,,,,
3,Q056,formal_paraphrase,ما المتطلبات المتعلقة بالارتباط بين الطلبات ال...,اللائحة التنفيذية لنظام المحاكم التجارية,المادة السابعة والسبعون,المادة السابعة والسبعون : بيانات صحيفةالدعوى\n...,[Retrieved Article 1]\nLaw: اللائحة التنفيذية ...,يجب أن تتضمن صحيفة الدعوى بيان الارتباط بين ال...,Model A,,,,,
4,Q027,informal,وش لازم أكتب في الإخطار اللي أرسله للطرف الثاني؟,اللائحة التنفيذية لنظام المحاكم التجارية,المادة السبعون,المادة السبعون:\nيجب أن يتضمن الإخطار بيانات ا...,[Retrieved Article 1]\nLaw: اللائحة التنفيذية ...,,Model A,,,,,


## 27. Final SANAD pipeline after Stage 3

**Stage 1**  
Saudi legal documents  
→ cleaning / legal article segmentation  
→ BGE-M3 embeddings  
→ FAISS vector index

**Stage 2 — Experiment B**  
User query  
→ BGE-M3 query embedding  
→ selected retrieval from Dense / BM25 / Hybrid  
→ Top-10

**Stage 2 — Experiment C**  
Top-10  
→ selected reranker from BGE / Qwen comparison  
→ Top-3

**Stage 3**  
Top-3 + user question  
→ augmented prompt  
→ selected generator from ALLaM / Qwen3 comparison  
→ grounded Arabic answer  
→ citation verification  
→ final answer with Saudi law name and article number

## 28. Save all Stage 3 outputs

In [ ]:
RESULTS_DIR = WORK_DIR / "final_results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

eval_df.to_csv(
    RESULTS_DIR / "SANAD_Stage3_Per_Query.csv",
    index=False,
    encoding="utf-8-sig",
)

automatic_summary_df.to_csv(
    RESULTS_DIR / "SANAD_Stage3_Automatic_Metrics.csv",
    index=False,
    encoding="utf-8-sig",
)

final_comparison_df.to_csv(
    RESULTS_DIR / "SANAD_Stage3_Model_Comparison.csv",
    index=False,
    encoding="utf-8-sig",
)

by_query_type_df.to_csv(
    RESULTS_DIR / "SANAD_Stage3_By_Query_Type.csv",
    index=False,
    encoding="utf-8-sig",
)

bootstrap_ci_df.to_csv(
    RESULTS_DIR / "SANAD_Stage3_Bootstrap_CI.csv",
    index=False,
    encoding="utf-8-sig",
)

shutil.copy(
    HUMAN_REVIEW_PATH,
    RESULTS_DIR / "SANAD_Stage3_Human_Review_Blinded.csv",
)

shutil.copy(
    CONTEXT_60_PATH,
    RESULTS_DIR / "SANAD_Stage3_Context_60.csv",
)

selection_output = pd.DataFrame([{
    "Selected_Retrieval": SELECTED_RETRIEVAL,
    "Selected_Reranker": SELECTED_RERANKER,
    "Selected_Generator": SELECTED_GENERATOR,
}])

selection_output.to_csv(
    RESULTS_DIR / "SANAD_Final_Pipeline_Selection.csv",
    index=False,
    encoding="utf-8-sig",
)

print("✅ All Stage 3 outputs saved in:")
print(RESULTS_DIR)

for p in sorted(RESULTS_DIR.iterdir()):
    print("-", p.name)

✅ All Stage 3 outputs saved in:
/content/drive/MyDrive/SANAD_Stage3_REBUILT/final_results
- SANAD_Final_Pipeline_Selection.csv
- SANAD_Stage3_Automatic_Metrics.csv
- SANAD_Stage3_Bootstrap_CI.csv
- SANAD_Stage3_By_Query_Type.csv
- SANAD_Stage3_Context_60.csv
- SANAD_Stage3_Human_Review_Blinded.csv
- SANAD_Stage3_Model_Comparison.csv
- SANAD_Stage3_Per_Query.csv
- SANAD_Stage3_Report_Summary.txt


## 29. Report-ready summary

In [ ]:
summary_lines = [
    "SANAD — Stage 3 Generation & Citation Verification",
    "",
    "Upstream decisions received from previous experiments:",
    f"Selected retrieval: {SELECTED_RETRIEVAL}",
    f"Selected reranker: {SELECTED_RERANKER}",
    "",
    f"Evaluation queries: {len(context_df)}",
    (
        "Gold article available in final Top-3: "
        f"{int(context_df['context_has_gold'].sum())}/{len(context_df)} "
        f"({context_df['context_has_gold'].mean():.2%})"
    ),
    "",
    "Generator comparison:",
]

for _, r in final_comparison_df.iterrows():
    summary_lines.extend([
        f"- {r['Model']}",
        (
            f"  Correctness: {r['Correctness_1_5']:.3f}"
            if pd.notna(r["Correctness_1_5"])
            else "  Correctness: N/A"
        ),
        (
            f"  Faithfulness: {r['Faithfulness_1_5']:.3f}"
            if pd.notna(r["Faithfulness_1_5"])
            else "  Faithfulness: N/A"
        ),
        f"  Citation Accuracy (all): {r['Citation_Accuracy_All']:.2%}",
        (
            "  Citation Accuracy (gold available): "
            f"{r['Citation_Accuracy_ContextHasGold']:.2%}"
        ),
        f"  Citation Precision: {r['Citation_Precision_Context']:.2%}",
        (
            f"  Hallucination Rate: {r['Hallucination_Rate']:.2%}"
            if pd.notna(r["Hallucination_Rate"])
            else "  Hallucination Rate: N/A"
        ),
        (
            f"  Arabic Quality: {r['Arabic_Quality_1_5']:.3f}"
            if pd.notna(r["Arabic_Quality_1_5"])
            else "  Arabic Quality: N/A"
        ),
        f"  Avg Generation Time: {r['Avg_Generation_Time_s']:.3f}s",
        "",
    ])

summary_lines.extend([
    f"Selected final generator: {SELECTED_GENERATOR}",
])

SUMMARY_PATH = RESULTS_DIR / "SANAD_Stage3_Report_Summary.txt"
SUMMARY_PATH.write_text(
    "\n".join(summary_lines),
    encoding="utf-8",
)

print("\n".join(summary_lines))
print("\nSaved:", SUMMARY_PATH)

SANAD — Stage 3 Generation & Citation Verification

Upstream decisions received from previous experiments:
Selected retrieval: Dense
Selected reranker: BAAI/bge-reranker-v2-m3

Evaluation queries: 60
Gold article available in final Top-3: 54/60 (90.00%)

Generator comparison:
- ALLaM-7B-Instruct-preview
  Correctness: 4.417
  Faithfulness: 4.150
  Citation Accuracy (all): 80.00%
  Citation Accuracy (gold available): 88.89%
  Citation Precision: 95.83%
  Hallucination Rate: 6.67%
  Arabic Quality: 4.667
  Avg Generation Time: 9.039s

- Qwen3-8B
  Correctness: 4.317
  Faithfulness: 3.967
  Citation Accuracy (all): 76.67%
  Citation Accuracy (gold available): 85.19%
  Citation Precision: 95.56%
  Hallucination Rate: 5.00%
  Arabic Quality: 4.567
  Avg Generation Time: 16.029s

Selected final generator: ALLaM-7B-Instruct-preview

Saved: /content/drive/MyDrive/SANAD_Stage3_REBUILT/final_results/SANAD_Stage3_Report_Summary.txt


## 30. Download final Stage 3 results ZIP

In [ ]:
from google.colab import files

ZIP_BASE = "/content/SANAD_Stage3_Results_REBUILT"

shutil.make_archive(
    ZIP_BASE,
    "zip",
    root_dir=str(RESULTS_DIR),
)

files.download(ZIP_BASE + ".zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>